# hs_code_check_v1 - HS 코드 앞자리 0 소실 점검 (읽기 전용)

**왜 점검하나**: `korea_company_hscode_map`, `target_company_db`, `target_hs_code`의 `hs_code`가 숫자형(bigint)입니다.
숫자로 저장하면 `0101...`(살아있는 동물), `0302...`(생선)처럼 0으로 시작하는 HS 코드의 앞자리 0이 사라집니다.
그러면 기업-HS 매핑과 수출입 테이블이 서로 연결되지 않아, 해당 기업의 드라이버를 찾지 못하게 됩니다.

**점검 내용**
1. 매핑 테이블의 HS 코드 자릿수 분포: 정상 HS 코드는 2·4·6·10자리(짝수). **홀수 자릿수면 앞자리 0이 사라진 것**
2. 수출입 테이블의 HS 코드 형태: 0으로 시작하는 코드가 실제로 있는지
3. 연결 테스트: 매핑 코드가 수출입 코드와 얼마나 맞는지, 앞자리 0을 되살리면 얼마나 더 맞는지
4. 텍스트형 HS 코드 테이블도 같은 문제가 있는지 (숫자형에서 옮겨 오며 0이 이미 빠졌을 수 있음)
5. (추가) 수출입 금액 컬럼이 FLOAT(정밀도 약 7자리)라 큰 금액이 반올림됐는지

DB는 읽기만 합니다. 결과 요약은 CSV로 저장됩니다 (분석용으로 업로드).

## 1. 설정 및 접속

In [1]:
import os
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
from sqlalchemy import text

# DATA 폴더 (config.py 위치). 없으면 현재 폴더에서 상위로 자동 탐색
DATA_DIR = r"/DATA"
# 결과 저장 위치
OUT_DIR = r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell"
SAVE_RESULT = True
# 450만 행 텍스트 테이블(correlation_price_hscode)까지 볼지. 1분 정도 더 걸릴 수 있음
CHECK_BIG_TEXT = True

data_dir = Path(DATA_DIR)
if not data_dir.exists():
    data_dir = next((p / "DATA" for p in [Path.cwd(), *Path.cwd().parents] if (p / "DATA").exists()), None)
if data_dir is None:
    raise FileNotFoundError("DATA 폴더를 찾지 못했습니다. DATA_DIR 경로를 확인하세요.")
for p in (str(data_dir), str(data_dir.parent)):
    if p not in sys.path:
        sys.path.insert(0, p)
import config
engine = config.get_engine(config.get_db_info())

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)


def q(sql, **params):
    with engine.connect() as cx:
        return pd.read_sql(text(sql), cx, params=params)


def table_exists(name):
    return not q("SELECT 1 FROM information_schema.TABLES WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = :t",
                 t=name).empty

print("DB 접속 완료:", engine.url.database)

DB 접속 완료: investar


## 2. 매핑 테이블(숫자형 hs_code)의 자릿수 분포
정상이면 짝수 자릿수(2·4·6·10)만 나옵니다. **홀수(1·3·5·9자리)가 보이면 앞자리 0이 사라진 코드**입니다.

In [2]:
MAP_TABLES = {   # 테이블: (종목코드 컬럼, 종목명 컬럼)
    "korea_company_hscode_map": ("ticker", "Name"),
    "target_company_db": ("Code", "Name"),
    "target_hs_code": (None, None),
}

len_rows, suspects = [], []
for t, (tk, nm) in MAP_TABLES.items():
    if not table_exists(t):
        print(f"[{t}] 테이블 없음 - 건너뜀")
        continue
    d = q(f"SELECT LENGTH(CAST(hs_code AS CHAR)) AS n_digits, COUNT(*) AS n_rows "
          f"FROM `{t}` WHERE hs_code IS NOT NULL GROUP BY n_digits ORDER BY n_digits")
    d.insert(0, "table", t)
    len_rows.append(d)
    cols = ", ".join(c for c in (tk, nm) if c)
    s = q(f"SELECT {cols + ', ' if cols else ''}CAST(hs_code AS CHAR) AS hs_code_stored "
          f"FROM `{t}` WHERE hs_code IS NOT NULL AND MOD(LENGTH(CAST(hs_code AS CHAR)), 2) = 1")
    if len(s):
        s["hs_code_restored"] = s["hs_code_stored"].apply(lambda x: x.zfill(len(x) + 1))
        s.insert(0, "table", t)
        suspects.append(s)

map_len = pd.concat(len_rows, ignore_index=True) if len_rows else pd.DataFrame()
map_len["판정"] = map_len["n_digits"].apply(lambda n: "의심(홀수 자릿수)" if n % 2 else "정상")
map_suspects = pd.concat(suspects, ignore_index=True) if suspects else pd.DataFrame()
print(map_len.to_string(index=False))
print(f"\n앞자리 0 소실 의심 행: {len(map_suspects)}개")
map_suspects.head(30)

                   table  n_digits  n_rows         판정
korea_company_hscode_map         4     140         정상
korea_company_hscode_map         5       6 의심(홀수 자릿수)
korea_company_hscode_map         6     332         정상
korea_company_hscode_map         9       1 의심(홀수 자릿수)
korea_company_hscode_map        10     283         정상
       target_company_db         3       1 의심(홀수 자릿수)
       target_company_db         4     145         정상
       target_company_db         5       6 의심(홀수 자릿수)
       target_company_db         6     340         정상
       target_company_db         9       1 의심(홀수 자릿수)
       target_company_db        10     315         정상
          target_hs_code         3       1 의심(홀수 자릿수)
          target_hs_code         4     145         정상
          target_hs_code         5       6 의심(홀수 자릿수)
          target_hs_code         6     340         정상
          target_hs_code         9       1 의심(홀수 자릿수)
          target_hs_code        10     315         정상

앞자리 0 소실 의심 행: 23개


,table,ticker,Name,hs_code_stored,hs_code_restored,Code
0,korea_company_hscode_map,A000210,DL,39022,039022,NaN
1,korea_company_hscode_map,A006040,동원산업,30341,030341,NaN
2,korea_company_hscode_map,A006040,동원산업,30342,030342,NaN
3,korea_company_hscode_map,A006040,동원산업,30343,030343,NaN
4,korea_company_hscode_map,A006040,동원산업,30344,030344,NaN
5,korea_company_hscode_map,A097950,CJ제일제당,19049,019049,NaN
6,korea_company_hscode_map,A267270,현대건설기계,842951101,0842951101,NaN
7,target_company_db,NaN,DL,39022,039022,A000210
8,target_company_db,NaN,동원산업,30341,030341,A006040
9,target_company_db,NaN,동원산업,30342,030342,A006040


## 3. 수출입 테이블의 HS 코드 형태
수출입 테이블에 0으로 시작하는 코드가 있는지, 자릿수가 어떻게 되는지 봅니다.

In [3]:
TRADE_TABLES = {   # 테이블: 코드 컬럼
    "korea_monthly_trade_data": "root_hs_code",
    "korea_monthly_trade_data_v2": "root_hs_code",
    "korea_quarterly_trade_data": "root_hs_code",
    "korea_monthly_trade_forecast_v2": "hs_code",
    "korea_regional_trade_data": "hs_code",
}
trade_codes, trade_shape = {}, []
for t, col in TRADE_TABLES.items():
    if not table_exists(t):
        continue
    codes = q(f"SELECT DISTINCT `{col}` AS code FROM `{t}`")["code"].dropna().astype(str).str.strip()
    trade_codes[t] = set(codes)
    lens = codes.str.len().value_counts().sort_index()
    trade_shape.append({
        "table": t, "n_codes": len(codes),
        "자릿수 분포": ", ".join(f"{k}자리 {v}개" for k, v in lens.items()),
        "0으로 시작": int(codes.str.startswith("0").sum()),
        "숫자 아닌 코드": int((~codes.str.fullmatch(r"\d+")).sum()),
        "예시": ", ".join(codes.sort_values().head(5)),
    })
trade_shape = pd.DataFrame(trade_shape)
trade_shape

,table,n_codes,자릿수 분포,0으로 시작,숫자 아닌 코드,예시
0,korea_monthly_trade_data,1168,"4자리 238개, 5자리 2개, 6자리 682개, 9자리 1개, 10자리 245개",6,0,"030354, 030389, 030487, 030633, 080610"
1,korea_monthly_trade_data_v2,498,6자리 498개,0,0,"121221, 151800, 190190, 190230, 190590"
2,korea_quarterly_trade_data,1002,"4자리 72개, 5자리 2개, 6자리 682개, 9자리 1개, 10자리 245개",6,0,"030354, 030389, 030487, 030633, 080610"
3,korea_monthly_trade_forecast_v2,1111,"4자리 235개, 5자리 2개, 6자리 643개, 9자리 1개, 10자리 230개",4,0,"030354, 030389, 030487, 030633, 1201"
4,korea_regional_trade_data,3,6자리 3개,0,0,"854231, 854232, 854239"


## 4. 연결 테스트: 매핑 코드 ↔ 수출입 코드
- `그대로 일치`: 지금 상태로 수출입 코드와 정확히 맞는 매핑 코드 수
- `0 복원 후 일치`: 홀수 자릿수 코드에 앞자리 0을 붙였더니 맞게 된 수 → **현재 연결이 끊겨 있는 실제 피해 규모**
- `앞부분 일치`: 정확히는 안 맞지만 수출입 코드가 매핑 코드로 시작하는 경우 (자릿수 단위 차이)

In [4]:
def restore(code_):
    return code_.zfill(len(code_) + 1) if len(code_) % 2 else code_


link = []
for mt in [t for t in MAP_TABLES if table_exists(t)]:
    m = q(f"SELECT DISTINCT CAST(hs_code AS CHAR) AS code FROM `{mt}` WHERE hs_code IS NOT NULL")["code"].astype(str)
    for tt, tcodes in trade_codes.items():
        exact = sum(c in tcodes for c in m)
        fixed = sum((c not in tcodes) and (restore(c) in tcodes) for c in m)
        prefix = sum((c not in tcodes) and (restore(c) not in tcodes) and any(x.startswith(restore(c)) for x in tcodes)
                     for c in m) if len(tcodes) < 50000 else None
        link.append({"매핑 테이블": mt, "수출입 테이블": tt, "매핑 코드 수": len(m),
                     "그대로 일치": exact, "0 복원 후 일치": fixed, "앞부분 일치": prefix,
                     "연결 안 됨": len(m) - exact - fixed - (prefix or 0)})
link = pd.DataFrame(link)
link

,매핑 테이블,수출입 테이블,매핑 코드 수,그대로 일치,0 복원 후 일치,앞부분 일치,연결 안 됨
0,korea_company_hscode_map,korea_monthly_trade_data,475,460,0,0,15
1,korea_company_hscode_map,korea_monthly_trade_data_v2,475,97,0,40,338
2,korea_company_hscode_map,korea_quarterly_trade_data,475,460,0,0,15
3,korea_company_hscode_map,korea_monthly_trade_forecast_v2,475,447,0,0,28
4,korea_company_hscode_map,korea_regional_trade_data,475,3,0,1,471
5,target_company_db,korea_monthly_trade_data,499,484,0,0,15
6,target_company_db,korea_monthly_trade_data_v2,499,100,0,40,359
7,target_company_db,korea_quarterly_trade_data,499,484,0,0,15
8,target_company_db,korea_monthly_trade_forecast_v2,499,469,0,0,30
9,target_company_db,korea_regional_trade_data,499,3,0,1,495


## 5. 텍스트형 HS 코드 테이블 점검
텍스트로 저장돼 있어도, 숫자형 데이터를 옮겨 오는 과정에서 앞자리 0이 이미 빠졌을 수 있습니다.

In [5]:
TEXT_TABLES = {
    "hs_code_by_kr_monster_company": ["hs_code", "hs_code_6d"],
    "correlation_table_DB": ["hs_code"],
    "korea_export_quick_report_data": ["hs_code"],
    "us_correlation_table": ["hs_code_6d"],
}
if CHECK_BIG_TEXT:
    TEXT_TABLES["correlation_price_hscode"] = ["hs_code"]

text_rows = []
for t, cols in TEXT_TABLES.items():
    if not table_exists(t):
        continue
    for col in cols:
        d = q(f"SELECT CHAR_LENGTH(TRIM(`{col}`)) AS n_digits, COUNT(DISTINCT `{col}`) AS n_codes, "
              f"SUM(TRIM(`{col}`) LIKE '0%') AS n_start0 FROM `{t}` WHERE `{col}` IS NOT NULL GROUP BY n_digits")
        odd = int(d.loc[d["n_digits"] % 2 == 1, "n_codes"].sum())
        text_rows.append({"table": t, "column": col, "코드 수": int(d["n_codes"].sum()),
                          "자릿수 분포": ", ".join(f"{int(a)}자리 {int(b)}" for a, b in zip(d["n_digits"], d["n_codes"])),
                          "홀수 자릿수 코드": odd, "0으로 시작(행)": int(d["n_start0"].sum()),
                          "판정": "의심" if odd else "정상"})
text_check = pd.DataFrame(text_rows)
text_check

,table,column,코드 수,자릿수 분포,홀수 자릿수 코드,0으로 시작(행),판정
0,hs_code_by_kr_monster_company,hs_code,184,"6자리 18, 9자리 1, 10자리 165",1,0,의심
1,hs_code_by_kr_monster_company,hs_code_6d,155,6자리 155,0,0,정상
2,correlation_table_DB,hs_code,3498,7자리 3498,3498,0,의심
3,korea_export_quick_report_data,hs_code,12,6자리 12,0,0,정상
4,us_correlation_table,hs_code_6d,478,6자리 478,0,0,정상
5,correlation_price_hscode,hs_code,1082,"4자리 238, 5자리 2, 6자리 614, 9자리 1, 10자리 227",3,0,의심


## 6. (추가) 수출입 금액 컬럼 정밀도
`korea_monthly_trade_data`, `korea_quarterly_trade_data`의 `value`가 FLOAT입니다. FLOAT는 유효숫자가 약 7자리라,
1,677만(2^24)을 넘는 값은 끝자리가 반올림됩니다 (예: 1,234,567,890 → 1,234,567,936).

In [6]:
prec = []
for t in ["korea_monthly_trade_data", "korea_quarterly_trade_data"]:
    if not table_exists(t):
        continue
    d = q(f"SELECT indicator, COUNT(*) AS n_rows, SUM(ABS(value) > 16777216) AS n_over_2p24, "
          f"MIN(value) AS min_value, MAX(value) AS max_value FROM `{t}` GROUP BY indicator")
    d.insert(0, "table", t)
    prec.append(d)
precision = pd.concat(prec, ignore_index=True) if prec else pd.DataFrame()
if len(precision):
    precision["반올림 가능 비율(%)"] = (precision["n_over_2p24"] / precision["n_rows"] * 100).round(1)
precision

,table,indicator,n_rows,n_over_2p24,min_value,max_value,반올림 가능 비율(%)
0,korea_monthly_trade_data,balPayments,243761,106684.0,-6.713070e+09,2.846740e+10,43.8
1,korea_monthly_trade_data,expDlr,243761,126833.0,0.000000e+00,3.310720e+10,52.0
2,korea_monthly_trade_data,expDlr_yoy,236698,0.0,-1.000000e+00,7.359080e+06,0.0
3,korea_monthly_trade_data,expWgt,114133,23457.0,0.000000e+00,5.163830e+09,20.6
4,korea_monthly_trade_data,impDlr,243761,89866.0,0.000000e+00,6.739000e+09,36.9
5,korea_monthly_trade_data,impDlr_yoy,236085,0.0,-1.000000e+00,1.428870e+06,0.0
6,korea_monthly_trade_data,impWgt,114133,8681.0,0.000000e+00,5.624980e+09,7.6
7,korea_quarterly_trade_data,balPayments,148021,68730.0,-1.701590e+10,2.425030e+10,46.4
8,korea_quarterly_trade_data,expDlr,148021,82572.0,0.000000e+00,3.360890e+10,55.8
9,korea_quarterly_trade_data,expDlr_yoy,145495,0.0,-1.000000e+00,7.359080e+06,0.0


## 7. 종합 판정 및 결과 저장

In [7]:
print("=" * 70)
n_sus = len(map_suspects)
print(f"1) 매핑 테이블 앞자리 0 소실 의심: {n_sus}행"
      + ("  → 문제 있음" if n_sus else "  → 문제 없음"))
if len(link):
    lost = link.groupby("매핑 테이블")["0 복원 후 일치"].max()
    for mt, v in lost.items():
        print(f"   - {mt}: 0을 되살리면 새로 연결되는 코드 {int(v)}개")
if len(trade_shape):
    z = int(trade_shape["0으로 시작"].sum())
    print(f"2) 수출입 테이블의 0으로 시작하는 코드: {z}개 "
          + ("(수출입 쪽은 0을 보존하고 있음)" if z else "(수출입 쪽에도 0 시작 코드가 없음 → 해당 품목군 미수집이거나 이미 소실)"))
if len(text_check):
    bad = text_check[text_check["판정"] == "의심"]
    print(f"3) 텍스트형 테이블 의심: {len(bad)}개 컬럼" + (f" → {', '.join(bad['table'] + '.' + bad['column'])}" if len(bad) else ""))
if len(precision):
    print(f"4) FLOAT 반올림 가능 행(1,677만 초과): {int(precision['n_over_2p24'].sum()):,}행 / {int(precision['n_rows'].sum()):,}행")
print("=" * 70)

if SAVE_RESULT:
    os.makedirs(OUT_DIR, exist_ok=True)
    out = os.path.join(OUT_DIR, f"hs_code_check_{STAMP}.xlsx")
    with pd.ExcelWriter(out, engine="openpyxl") as w:
        for name, df_ in [("map_length", map_len), ("map_suspects", map_suspects), ("trade_shape", trade_shape),
                          ("link_test", link), ("text_tables", text_check), ("float_precision", precision)]:
            (df_ if len(df_) else pd.DataFrame({"note": ["no rows"]})).to_excel(w, sheet_name=name, index=False)
    print("결과 저장:", out)

1) 매핑 테이블 앞자리 0 소실 의심: 23행  → 문제 있음
   - korea_company_hscode_map: 0을 되살리면 새로 연결되는 코드 0개
   - target_company_db: 0을 되살리면 새로 연결되는 코드 0개
   - target_hs_code: 0을 되살리면 새로 연결되는 코드 0개
2) 수출입 테이블의 0으로 시작하는 코드: 16개 (수출입 쪽은 0을 보존하고 있음)
3) 텍스트형 테이블 의심: 3개 컬럼 → hs_code_by_kr_monster_company.hs_code, correlation_table_DB.hs_code, correlation_price_hscode.hs_code
4) FLOAT 반올림 가능 행(1,677만 초과): 581,955행 / 2,244,531행
결과 저장: C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell\hs_code_check_20261001_2059.xlsx
